# Hotel Booking Cancellation — EDA & Feature Engineering


**Target:** `is_canceled`

**Assigned columns:** `days_in_waiting_list`, `customer_type`, `adr`, `required_car_parking_spaces`, `total_of_special_requests`, `reservation_status`, `reservation_status_date`

Analysis is focused on useful EDA, target relationships, data quality, leakage, and justified feature engineering. Random feature-feature plots are intentionally avoided.

## 1. Import Libraries and Load Dataset

In [ ]:
df = pd.read_csv('Hotel_booking_demand.csv')


## 2. Data Quality Check

In [ ]:
quality_check = pd.DataFrame({
    'Column': sanji_cols,
    'Data Type': [df[c].dtype for c in sanji_cols],
    'Missing Values': [df[c].isna().sum() for c in sanji_cols],
    'Missing %': [df[c].isna().mean()*100 for c in sanji_cols],
    'Unique Values': [df[c].nunique() for c in sanji_cols]
})
quality_check

## 3. `days_in_waiting_list` — Univariate

In [ ]:
df['days_in_waiting_list'].describe()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['days_in_waiting_list'], bins=50)
plt.title('Distribution of Days in Waiting List')
plt.xlabel('Days in Waiting List')
plt.ylabel('Frequency')
plt.show()

In [ ]:
q1 = df['days_in_waiting_list'].quantile(.25)
q3 = df['days_in_waiting_list'].quantile(.75)
iqr = q3-q1
lower, upper = q1-1.5*iqr, q3+1.5*iqr
outliers = df[(df['days_in_waiting_list']<lower)|(df['days_in_waiting_list']>upper)]
print('Lower bound:', lower)
print('Upper bound:', upper)
print('IQR outliers:', len(outliers))

**Observation:** Most bookings have 0 waiting-list days, so the distribution is strongly right-skewed. A small number have unusually long waiting periods.

## 4. `days_in_waiting_list` → `is_canceled`

In [ ]:
df['waiting_list_group'] = pd.cut(
    df['days_in_waiting_list'], [-1,0,7,30,90,np.inf],
    labels=['0','1-7','8-30','31-90','91+']
)
waiting_cancel = df.groupby('waiting_list_group', observed=False)['is_canceled'].agg(['count','mean'])
waiting_cancel['cancellation_rate_%'] = waiting_cancel['mean']*100
waiting_cancel.drop(columns='mean')

In [ ]:
waiting_cancel['cancellation_rate_%'].plot(kind='bar', figsize=(8,4))
plt.title('Cancellation Rate by Waiting List Group')
plt.xlabel('Waiting List Group')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=0)
plt.show()

**Observation:** Bookings with waiting-list days generally have higher cancellation rates than bookings with zero waiting-list days. The relationship is not perfectly monotonic at very high waiting times.

## 5. `customer_type` — Univariate and Target Relationship

In [ ]:
display(df['customer_type'].value_counts().to_frame('count'))
display((df['customer_type'].value_counts(normalize=True)*100).round(2).to_frame('percentage'))

df['customer_type'].value_counts().plot(kind='bar', figsize=(8,4))
plt.title('Distribution of Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=0)
plt.show()

**Observation:** `Transient` customers dominate the dataset, while `Group` bookings are relatively rare.

In [ ]:
customer_cancel = df.groupby('customer_type')['is_canceled'].agg(['count','mean'])
customer_cancel['cancellation_rate_%'] = customer_cancel['mean']*100
customer_cancel = customer_cancel.sort_values('cancellation_rate_%', ascending=False)
customer_cancel.drop(columns='mean')

In [ ]:
customer_cancel['cancellation_rate_%'].plot(kind='bar', figsize=(8,4))
plt.title('Cancellation Rate by Customer Type')
plt.xlabel('Customer Type')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=0)
plt.show()

**Observation:** Cancellation behavior differs substantially by customer type. Transient customers have the highest cancellation rate, while Group bookings have the lowest.

## 6. `adr` — Univariate and Target Relationship

In [ ]:
df['adr'].describe()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['adr'], bins=50)
plt.title('Distribution of ADR')
plt.xlabel('ADR')
plt.ylabel('Frequency')
plt.show()

q1=df['adr'].quantile(.25); q3=df['adr'].quantile(.75); iqr=q3-q1
lower=q1-1.5*iqr; upper=q3+1.5*iqr
print('IQR outliers:', ((df['adr']<lower)|(df['adr']>upper)).sum())
print('Negative ADR:', (df['adr']<0).sum())
print('ADR > 1000:', (df['adr']>1000).sum())

In [ ]:
df.loc[(df['adr']<0)|(df['adr']>1000), ['adr','hotel','arrival_date_year','arrival_date_month','arrival_date_day_of_month','is_canceled']]

**Observation:** ADR is strongly right-skewed and contains unusual values, including a negative value and an extremely high value. These should be investigated before deciding how to treat them.

In [ ]:
df['adr_group'] = pd.cut(df['adr'], [-np.inf,0,50,100,150,200,300,np.inf],
    labels=['<0','0-50','50-100','100-150','150-200','200-300','300+'])
adr_cancel = df.groupby('adr_group', observed=False)['is_canceled'].agg(['count','mean'])
adr_cancel['cancellation_rate_%'] = adr_cancel['mean']*100
adr_cancel.drop(columns='mean')

In [ ]:
adr_cancel['cancellation_rate_%'].plot(kind='bar', figsize=(9,4))
plt.title('Cancellation Rate by ADR Range')
plt.xlabel('ADR Range')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=0)
plt.show()

**Observation:** Cancellation varies across ADR ranges, but there is no simple linear relationship. A nonlinear representation may be useful.

## 7. `required_car_parking_spaces` — Univariate and Target Relationship

In [ ]:
parking_counts=df['required_car_parking_spaces'].value_counts().sort_index()
display(parking_counts.to_frame('count'))
display((parking_counts/len(df)*100).round(2).to_frame('percentage'))

parking_counts.plot(kind='bar', figsize=(8,4))
plt.title('Distribution of Required Car Parking Spaces')
plt.xlabel('Parking Spaces')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=0)
plt.show()

**Observation:** Approximately 94% of bookings require no parking. Non-zero values are rare.

In [ ]:
parking_cancel=df.groupby('required_car_parking_spaces')['is_canceled'].agg(['count','mean'])
parking_cancel['cancellation_rate_%']=parking_cancel['mean']*100
parking_cancel.drop(columns='mean')

**Observation:** Non-zero parking categories show very low cancellation rates, but their sample sizes are very small. We should not overinterpret this pattern.

## 8. `total_of_special_requests` — Univariate and Target Relationship

In [ ]:
special_counts=df['total_of_special_requests'].value_counts().sort_index()
display(special_counts.to_frame('count'))
display((special_counts/len(df)*100).round(2).to_frame('percentage'))

special_counts.plot(kind='bar', figsize=(8,4))
plt.title('Distribution of Total Special Requests')
plt.xlabel('Number of Special Requests')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=0)
plt.show()

**Observation:** Most bookings have 0 or 1 special request; 4–5 requests are uncommon.

In [ ]:
special_cancel=df.groupby('total_of_special_requests')['is_canceled'].agg(['count','mean'])
special_cancel['cancellation_rate_%']=special_cancel['mean']*100
special_cancel.drop(columns='mean')

In [ ]:
special_cancel['cancellation_rate_%'].plot(kind='bar', figsize=(8,4))
plt.title('Cancellation Rate by Number of Special Requests')
plt.xlabel('Number of Special Requests')
plt.ylabel('Cancellation Rate (%)')
plt.xticks(rotation=0)
plt.show()

**Observation:** Bookings with no special requests have a substantially higher cancellation rate. Cancellation generally decreases as special requests increase.

## 9. `reservation_status` — Leakage Investigation

In [ ]:
display(df['reservation_status'].value_counts().to_frame('count'))
status_target=pd.crosstab(df['reservation_status'],df['is_canceled'],normalize='index')*100
status_target.round(2)

### 🚨 Critical finding

`reservation_status` almost perfectly reveals `is_canceled`. If prediction is made at booking time, the final reservation status is not available yet.

**Decision: exclude `reservation_status` from the model.**

## 10. `reservation_status_date` — Leakage Investigation

In [ ]:
df['reservation_status_date']=pd.to_datetime(df['reservation_status_date'])
print('Minimum date:',df['reservation_status_date'].min())
print('Maximum date:',df['reservation_status_date'].max())
print('Unique dates:',df['reservation_status_date'].nunique())

In [ ]:
status_date_monthly=(df['reservation_status_date'].dt.to_period('M').value_counts().sort_index())
plt.figure(figsize=(12,4))
status_date_monthly.plot()
plt.title('Reservation Status Updates Over Time')
plt.xlabel('Month')
plt.ylabel('Number of Records')
plt.show()

### 🚨 Critical finding

`reservation_status_date` is post-booking information. It would not be available at the intended prediction point.

**Decision: exclude `reservation_status_date` from the model.**

# 11. Feature Engineering

Feature engineering is based on the EDA findings. We will create candidate features only where there is a clear reason.

In [ ]:
# 1. Waiting-list features
df['has_waiting_list']=(df['days_in_waiting_list']>0).astype(int)
df['waiting_list_category']=pd.cut(
    df['days_in_waiting_list'],[-1,0,7,30,90,np.inf],
    labels=['None','Short','Medium','Long','Very_Long']
)

# 2. Customer type encoding
customer_dummies=pd.get_dummies(df['customer_type'],prefix='customer_type',drop_first=True,dtype=int)

# 3. ADR candidate cleaning and transformations
df['adr_clean']=df['adr'].copy()
df.loc[df['adr_clean']<0,'adr_clean']=np.nan
df['adr_log']=np.log1p(df['adr_clean'])
df['adr_category']=pd.cut(
    df['adr_clean'],[-np.inf,50,100,150,200,300,np.inf],
    labels=['Very_Low','Low','Medium','High','Very_High','Premium']
)

# 4. Parking feature
df['requires_parking']=(df['required_car_parking_spaces']>0).astype(int)

# 5. Special-request features
df['has_special_request']=(df['total_of_special_requests']>0).astype(int)
df['special_request_category']=pd.cut(
    df['total_of_special_requests'],[-1,0,1,2,np.inf],
    labels=['None','One','Two','Three_or_More']
)

print('Candidate feature engineering completed.')

## 12. Review Candidate Engineered Features

In [ ]:
candidate_features=[
    'days_in_waiting_list','has_waiting_list','waiting_list_category',
    'customer_type','adr','adr_clean','adr_log','adr_category',
    'required_car_parking_spaces','requires_parking',
    'total_of_special_requests','has_special_request','special_request_category'
]

display(df[candidate_features].head())
print('\nCustomer dummy columns:')
print(customer_dummies.columns.tolist())

## 13. Remove Leakage Features

In [ ]:
leakage_cols=['reservation_status','reservation_status_date']
df_model=df.drop(columns=leakage_cols)

print('Removed:', leakage_cols)
print('Modeling dataset shape:', df_model.shape)

# 14. Feature Engineering Validation Strategy

Do not automatically keep every engineered feature.

Compare feature sets using the same train/validation strategy and evaluation metric:

1. Baseline — original features
2. Baseline + `has_waiting_list`
3. Baseline + `has_special_request`
4. Baseline + `requires_parking`
5. Baseline + selected ADR transformation
6. Baseline + selected engineered features

The final decision should be based on validation performance, model stability, interpretability, and absence of leakage.

# 15. Final Findings

### Strong candidate predictors
- `customer_type`
- `days_in_waiting_list`
- `total_of_special_requests`
- `adr`

### Candidate engineered features
- `has_waiting_list`
- `waiting_list_category`
- One-hot encoded `customer_type`
- `adr_clean`
- `adr_log`
- `adr_category`
- `requires_parking`
- `has_special_request`
- `special_request_category`

### Exclude from predictive modeling
- `reservation_status` — target leakage
- `reservation_status_date` — post-booking information

